In [ ]:
#Checking splitting in Q(sqrt(5)) for d=5 cases
k.<sqrt5> = QuadraticField(5)
print(kronecker(k.disc(),1201))
k.factor(461)

In [22]:
#Split in Q(sqrt(5)) for d=5 cases
K.<a> = QuadraticField(5)
Z_sqrt5 = K.order(a)
u = (1 + a) / 2

# Take the first prime factor from K.ideal(281).factor()
P = K.ideal(-1559).factor()[0][0]
g1 = P.gens_reduced()[0]

# Shift by powers of the fundamental unit u until g1 is in Z[sqrt(5)]
while g1 not in Z_sqrt5:
    g1 *= u

# Get its exact algebraic conjugate
g2 = g1.galois_conjugate()
print(g1,g2)
# part with a, is Ln, other is Fn

20*a + 21 -20*a + 21


In [27]:
#FULLY SELF CONTAINED CHECKING OF SEQUENCES MOD PRIMES THAT ARE CONGRUENT TO 1 MOD 11
from itertools import combinations
import pandas as pd

def calculate_xn_mod(a, b, max_n):
    """
    Calculates x_n = a*F_n + b*L_n modulo a specific prime up to max_n
    """
    # Base states for Fibonacci (F) and Lucas (L)
    f_prev, f_curr = 0, 1
    l_prev, l_curr = 2, 1
    
    xn_mod_sequence = []
    
    # Calculate n = 0 manually
    x0 = (a * f_prev + b * l_prev) 
    xn_mod_sequence.append(x0)
    
    # Calculate n = 1 manually
    x1 = (a * f_curr + b * l_curr) 
    xn_mod_sequence.append(x1)
    
    # Loop for subsequent terms
    for n in range(2, max_n + 1):
        # Compute next elements in the base sequences
        f_next = (f_prev + f_curr) 
        l_next = (l_prev + l_curr) 
        
        # Compute x_n modulo m
        xn = (a * f_next + b * l_next) 
        xn_mod_sequence.append(xn)
        
        # Step forward in history
        f_prev, f_curr = f_curr, f_next
        l_prev, l_curr = l_curr, l_next
        
    return xn_mod_sequence



def check_perfect_11(x,prime,exponent): # check if a number is a perfect 11th power mod p
    element_mod_p = Mod(x,prime)
    return pari.ispower(element_mod_p, exponent)

# TO-DO create period length computing function
def compute_period_length(a, b, modulus):
    """
    Computes the period length of the sequence x_n = a*F_n + b*L_n modulo modulus
    Returns the period length and the sequence values
    """
    # Generate sequence until we find a repeat of (x_0, x_1)
    f_prev, f_curr = 0, 1  # F_0, F_1
    l_prev, l_curr = 2, 1  # L_0, L_1
    
    # Calculate initial values
    x0 = (a * f_prev + b * l_prev) % modulus
    x1 = (a * f_curr + b * l_curr) % modulus
    
    # Store pairs (F_n, L_n) or directly store (x_n, x_{n+1})
    sequence_pairs = [(f_prev, l_prev), (f_curr, l_curr)]
    xn_sequence = [x0, x1]
    
    # Keep track of seen states: (F_n, L_n) -> index
    seen_states = {(f_prev, l_prev): 0, (f_curr, l_curr): 1}
    
    n = 1  # We already have n=0 and n=1
    while True:
        n += 1
        
        # Compute next elements
        f_next = (f_prev + f_curr) % modulus
        l_next = (l_prev + l_curr) % modulus
        
        state = (f_next, l_next)
        
        # Check if we've seen this state before
        if state in seen_states:
            # We found the period
            start_index = seen_states[state]
            period_length = n - start_index
            
            # Return all sequence values up to the period
            return period_length
        
        # Store the new state
        seen_states[state] = n
        sequence_pairs.append(state)
        
        # Compute x_n
        xn = (a * f_next + b * l_next) % modulus
        xn_sequence.append(xn)
        
        # Step forward
        f_prev, f_curr = f_curr, f_next
        l_prev, l_curr = l_curr, l_next

def get_eleventh_powers(a,b,primes, max_prime_index):
    power_indices = dict() # to keep which n are perfect 11th powers mod a prime
    primes_to_period = dict() # track period of sequence mod prime
    for p in primes[:max_prime_index]:
        period = compute_period_length(a,b,p)
        seq = calculate_xn_mod(a,b,period-1)
        indices = []
        for el in seq:
            if check_perfect_11(el, p, 11):
                indices.append(seq.index(el))
        if len(indices) == 0:
            print("No 11th powers at", p)
            break
        power_indices[period] = indices
        primes_to_period[period] = p
        print("prime: ", p, "period: ", period)
    print("Done computing 11th powers")
    return power_indices, primes_to_period

def chinese_remainder_thm(x,y,m,n):
    if (x - y) % gcd(m,n) == 0:
        return True
    else:
        return False

def check_indices(perfect_powers):
    for key in perfect_powers:
        perfect_powers[key] = [index for index in perfect_powers[key] if index % 2 == 0]
    for key_pair in combinations(perfect_powers, 2):
        print("Working on: ", key_pair)
        if not any(chinese_remainder_thm(n,m, key_pair[0], key_pair[1]) for n in perfect_powers[key_pair[0]] for m in perfect_powers[key_pair[1]]):
            print("Periods of length", key_pair[0], "and", key_pair[1], "do not work")
            return key_pair[0],key_pair[1]
    print('done')

#coefficients of the sequence
a = 31
b = 10

# primes congruent to 1 mod 11
primes = Primes(modulus=11, classes=[1])

perfect_power_indices, period_of_primes = get_eleventh_powers(a,b,primes,10)
print(perfect_power_indices)
check_indices(perfect_power_indices)



prime:  23 period:  48
prime:  67 period:  136
prime:  89 period:  44
prime:  199 period:  22
prime:  331 period:  110
prime:  353 period:  236
prime:  397 period:  796
prime:  419 period:  418
prime:  463 period:  928
prime:  617 period:  1236
Done computing 11th powers
{48: [18, 19, 20, 21, 22, 42, 43, 44, 45, 46], 136: [4, 13, 25, 30, 35, 47, 56, 62, 66, 72, 81, 93, 98, 103, 115, 124, 130, 134], 44: [9, 20, 31, 42], 22: [20, 21], 110: [7, 8, 17, 38, 63, 73, 75, 102, 108], 236: [9, 16, 33, 35, 50, 57, 68, 75, 92, 94, 109, 116, 127, 134, 151, 153, 168, 175, 186, 193, 210, 212, 227, 234], 796: [1, 38, 39, 43, 44, 54, 76, 98, 108, 109, 113, 114, 151, 154, 197, 200, 237, 238, 242, 243, 253, 275, 297, 307, 308, 312, 313, 350, 353, 396, 399, 436, 437, 441, 442, 452, 474, 496, 506, 507, 511, 512, 549, 552, 595, 598, 635, 636, 640, 641, 651, 673, 695, 705, 706, 710, 711, 748, 751, 794], 418: [8, 21, 34, 44, 49, 62, 64, 76, 78, 79, 82, 86, 93, 104, 107, 113, 123, 143, 144, 149, 157, 166, 189,